# Introduction to DSPy

This notebook provides a comprehensive guide to using **DSPy**, a framework for algorithmically optimizing Language Model (LM) prompts and weights. 
We'll cover basic concepts including:
- LLM Configuration
- Signatures and Predictors
- Custom Modules (Chaining logic)
- ReAct Agents with Tools
- Optimizers (Teleprompters) for prompt tuning

In [ ]:
# Import required libraries
import dspy
import os
from dotenv import load_dotenv
from pprint import pprint

# Load environment variables (e.g., GROQ_API_KEY)
load_dotenv()

## 1. Setting up the Language Model

First, we configure DSPy to use a specific LLM. In this tutorial, we're using the `llama-3.3-70b-versatile` model hosted on Groq for fast inference.

In [ ]:
# Initialize the LLM via Groq
llm = dspy.LM(model="groq/llama-3.3-70b-versatile")

# Configure DSPy to use this LLM globally
dspy.configure(lm=llm)

### Basic Direct Call
You can call the LLM directly with a list of messages, bypassing DSPy's more advanced features. This is similar to the standard OpenAI API.

In [ ]:
messages = [
    {
        "role": "system",
        "content": "You are a helpful assistant"
    },
    {
        "role": "user",
        "content": "What is the capital of France?"
    }
]

# Execute the direct LLM call
pprint(llm(messages = messages))

## 2. DSPy Signatures and Predictors

**Signatures** define the declarative input/output behavior of a module. 
The `dspy.Predict` module is the most basic building block that takes a signature and predicts the output.

In [ ]:
# Define a simple signature: given a subject, generate a haiku
haiku_signature = "subject -> haiku"

# Initialize the predictor
haiku_generator = dspy.Predict(haiku_signature)

# Run the predictor
result = haiku_generator(subject="computer science")
print(result.haiku)

We can easily change the signature to produce a different type of output, such as a limerick, without changing how we invoke the module.

In [ ]:
# Modify the signature to generate a limerick instead
haiku_signature = "subject -> limerick"
haiku_generator = dspy.Predict(haiku_signature)

result = haiku_generator(subject="computer science")
print(result.limerick)

Signatures can also include complex types (like lists) and boolean flags to guide the generation.

In [ ]:
# Signature with multiple inputs and a complex output type (list of strings)
haiku_bot = dspy.Predict("location, mood, contains_pun: bool -> haikus: list[str]")

# Generate multiple haikus based on the parameters
result = haiku_bot(location="a quiet library", mood="mysterious", contains_pun=True)
result

### Class-based Signatures
For more complex or verbose prompts, you can define a class inheriting from `dspy.Signature`. This allows you to add detailed descriptions (`desc`) to input and output fields, acting as detailed prompt instructions.

In [ ]:
from typing import Literal

class HaikuBot(dspy.Signature):
    """Generates a haiku based on location, mood, and season."""
    
    # Input fields with optional descriptions
    location: str = dspy.InputField(desc="The setting of the poem")
    mood: str = dspy.InputField()
    season: Literal["spring", "summer", "autumn", "winter"] = dspy.InputField()
    
    # Output field
    haiku: str = dspy.OutputField()

In [ ]:
# Instantiate a predictor using the class-based signature
bot = dspy.Predict(HaikuBot)

# Run the bot
res = bot(location="Bodega Bay", mood="mysterious", season="autumn")
res

## 3. DSPy Modules (Chaining)

Modules in DSPy allow you to compose multiple predictors together. 
Here we build an ensemble module (`HaikuEnsemble`) that:
1. Generates multiple candidate haikus.
2. Uses a `dspy.ChainOfThought` judge to evaluate and select the best one.

In [ ]:
class HaikuEnsemble(dspy.Module):
    def __init__(self, num_candidates: int = 3):
        super().__init__()
        self.num_candidates = num_candidates
        
        # Module 1: Generates a list of candidate haikus
        self.writer = dspy.Predict("location, season -> haikus: list[str]")
        
        # Module 2: ChainOfThought Judge to pick the best candidate
        # ChainOfThought automatically adds a 'reasoning' field before the final output
        self.judge = dspy.ChainOfThought("location, season, candidates: list[str] -> best_index: int")

    def forward(self, location: str, season: str):
        # Step 1: Draft multiple candidates
        candidates = self.writer(location=location, season=season).haikus
        
        # Step 2: Judge picks the best index based on reasoning
        verdict = self.judge(location=location, season=season, candidates=candidates)
        
        return dspy.Prediction(
            haiku=candidates[verdict.best_index],
            reasoning=verdict.reasoning
        )

In [ ]:
# Run the composite ensemble module
ensemble = HaikuEnsemble()
result = ensemble(location="Tokyo", season="spring")

# Display the reasoning process and the final selected haiku
print("Reasoning:\n", result.reasoning)
print("\nSelected Haiku:\n", result.haiku)

## 4. ReAct Agents and Tools

DSPy supports ReAct (Reasoning and Acting) agents that can use external python functions as tools to gather information before generating an answer.

In [ ]:
import wikipedia

# Define tool functions with proper type hints and docstrings
# DSPy uses docstrings to understand what the tool does
def wikipedia_search(query: str) -> list[str]:
    """Search Wikipedia for the given query and return a list of page titles."""
    return wikipedia.search(query)

def get_wikipedia_page(title: str) -> str:
    """Get the content of a Wikipedia page given its title."""
    return wikipedia.page(title).content

In [ ]:
# Create the ReAct agent
# The agent will interleave thoughts, tool calls, and observations
agent = dspy.ReAct(
    "location, topic -> poem", 
    tools=[wikipedia_search, get_wikipedia_page], 
    max_iters=5
)

# Run the agent
# Note: This might encounter Wikipedia API errors depending on connectivity,
# but demonstrates the ReAct paradigm.
result = agent(location="San Francisco", topic="Golden Gate Bridge history")

In [ ]:
result

## 5. DSPy Optimizers (Teleprompters)

Optimizers algorithmically tune the prompts or weights of your DSPy programs. 
We'll use the **GEPA optimizer**, which uses an LLM to reflect and rewrite instructions based on a metric.

In [ ]:
# Define training examples with gold inputs
examples = [
    dspy.Example(location="Paris", season="spring").with_inputs("location", "season"),
    dspy.Example(location="New York", season="winter").with_inputs("location", "season")
]

We define a custom metric function. The metric function takes the example (gold), prediction, and other trace context. It returns a float score (e.g., 0.0 to 1.0).

In [ ]:
# Metric function to evaluate the prediction
# GEPA requires exactly 5 arguments for the metric signature
def haiku_score(example, prediction, trace=None, pred_name=None, pred_trace=None) -> float:
    text = prediction.haiku.lower()
    
    # Penalize (score=0.0) if the haiku mentions the season word explicitly
    if example.season.lower() in text:
        return 0.0
        
    # Reward (score=1.0) otherwise
    return 1.0

In [ ]:
# 1. Choose an optimizer (GEPA uses an LLM to reflect and rewrite prompts)
optimizer = dspy.GEPA(
    metric=haiku_score,
    reflection_lm=dspy.LM("groq/llama-3.3-70b-versatile"), # Use a smart model as a teacher
    auto="light"
)

# 2. Compile!
# This is where DSPy simulates runs, checks the metric, and rewrites instructions.
optimized_bot = optimizer.compile(bot, trainset=examples)